# Workflow: Routing

## Load Keys in Env.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from cohere import ClientV2

co = ClientV2(api_key=os.environ["COHERE_API_KEY"])


In [3]:
MODEL_NAME = "command-a-03-2025"

## Define System Prompts for Routes

In [4]:
ROUTES = {
    "billing": "You are a billing specialist. Be precise about amounts, dates, and refund policies. Always end with: 'Anything else I can help with on your account?",
    "technical": "You are a senior support engineer. Diagnose technical issues step by step. Ask one clarifying question if needed. Be concise.",
    "general": "You are a friendly first0line support rep. Answer warmly and briefly."
}

## Define Helper for LLM calls

In [ ]:
def llm(prompt: str, system: str = "") -> str:
    msgs = []
    if system:
        msgs.append(
            {
                "role": "system",
                "content": system,
            }
        )
    msgs.append(
        {
            "role": "user",
            "content": prompt,
        }
    )

    return (
        co.chat(
            model=MODEL_NAME,
            messages=msgs,
        )
        .message.content[0]
        .text.strip()
    )

## Define Router

In [6]:
def route(user_msg: str) -> str:
    label = (
        llm(
            f"Classify this support message into exactly one of: {", ".join(ROUTES.keys())}.\n\n"
            f"Message: {user_msg!r}\nRETURN ONLY the label, lowercase, no punctuation."
        )
        .lower()
        .strip()
    )

    return label if label in ROUTES else "general"

## Call The Identified Route

In [14]:
def handle(user_msg: str) -> str:
    label = route(user_msg)
    answer = llm(user_msg, system=ROUTES[label])
    return f"[ROUTED TO {label}]\n\n{answer}"

## Test Routing

In [15]:
for msg in [
    "Why was I charged twice this month?",
    "The app crashes whenever I open the analytics tab.",
    "Hey, just wanted to say I love your product!",
]:
    print("USER:", msg)
    print(handle(msg))
    print("-" * 60)

USER: Why was I charged twice this month?
[ROUTED TO billing]

I’m sorry to hear about the double charge. Let me look into this for you. Could you please provide the exact dates of the charges and the amounts? This will help me investigate further and ensure the issue is resolved promptly.  

Once I have the details, I’ll check if the charges were for separate services, if there was an error, or if one of them is a pending authorization. If it’s a duplicate charge, I’ll process a refund immediately, typically within 5–7 business days, depending on your payment method.  

Anything else I can help with on your account?
------------------------------------------------------------
USER: The app crashes whenever I open the analytics tab.
[ROUTED TO technical]

To diagnose the issue, let's start with the following steps:

1. **Check Logs**: Look for error logs or crash reports in the app's log files or crash reporting tools (e.g., Firebase Crashlytics, Sentry).

2. **Reproduce the Issue**: C

## Routing vs. a single mega-prompt

You could write one prompt that says "if it's a billing question, be precise; if it's technical, diagnose; otherwise, be warm". It will mostly work. But:

- You can't test each branch independently.
- The model spends tokens reading instructions for branches that don't apply.
- Adding a fourth category means re-tuning the whole prompt.


> Use a small/cheap model for the classifier, the big model only for the specialist call. Classification is easier than generation

> Always have a safe default